In [1]:
import os
import glob
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt
from IPython.display import clear_output

In [2]:
class CorneaDataset(Dataset):
    def __init__(self, images_list, masks_list, transform=None):
        """
        images_list: لیستی از مسیرهای تصاویر اصلی
        masks_list:  لیستی از مسیرهای ماسک‌های مربوط به قرنیه
        transform:   ترنسفورم‌های مورد استفاده؛ اینجا اگر خواستید Augmentation اضافه کنید
        """
        self.images_list = images_list
        self.masks_list = masks_list
        self.transform = transform

    def __len__(self):
        return len(self.images_list)

    def __getitem__(self, idx):
        # خواندن تصویر
        img_path = self.images_list[idx]
        image = Image.open(img_path).convert("RGB")  # یا "L" اگر تصاویر سیاه‌وسفید هستند

        # خواندن ماسک
        mask_path = self.masks_list[idx]
        mask = Image.open(mask_path).convert("L")   # ماسک معمولاً تک کاناله است

        # تبدیل به Tensor
        # اینجا برای سادگی از یک ترنسفورم ساده ToTensor استفاده می‌کنیم
        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)
        else:
            image = transforms.ToTensor()(image)
            mask = transforms.ToTensor()(mask)

        # ماسک باید باینری باشد (0 یا 1)
        # گاهی ممکن است ماسک در محدوده [0,255] باشد
        mask = (mask > 0.5).float()

        return image, mask

In [3]:
class DoubleConv(nn.Module):
    """
    بلوک پایه‌ای U-Net که شامل دو کانولوشن پشت‌سرهم است
    """
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),                      
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

In [4]:
class UNet(nn.Module):
    """
    معماری ساده U-Net برای سگمنت کردن یک کانال خروجی (ماسک باینری)
    """
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()

        # Down (Encoder)
        self.conv_down1 = DoubleConv(in_channels, 64)
        self.conv_down2 = DoubleConv(64, 128)
        self.conv_down3 = DoubleConv(128, 256)
        self.conv_down4 = DoubleConv(256, 512)

        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Bottleneck
        self.bottleneck = DoubleConv(512, 1024)

        # Up (Decoder)
        self.uptrans1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv_up1 = DoubleConv(1024, 512)

        self.uptrans2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv_up2 = DoubleConv(512, 256)

        self.uptrans3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv_up3 = DoubleConv(256, 128)

        self.uptrans4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv_up4 = DoubleConv(128, 64)

        # Output
        self.output = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # Down
        x1 = self.conv_down1(x)
        x2 = self.maxpool(x1)

        x2 = self.conv_down2(x2)
        x3 = self.maxpool(x2)

        x3 = self.conv_down3(x3)
        x4 = self.maxpool(x3)

        x4 = self.conv_down4(x4)
        x5 = self.maxpool(x4)

        # Bottleneck
        x5 = self.bottleneck(x5)

        # Up
        x6 = self.uptrans1(x5)
        x6 = torch.cat([x4, x6], dim=1)
        x6 = self.conv_up1(x6)

        x7 = self.uptrans2(x6)
        x7 = torch.cat([x3, x7], dim=1)
        x7 = self.conv_up2(x7)

        x8 = self.uptrans3(x7)
        x8 = torch.cat([x2, x8], dim=1)
        x8 = self.conv_up3(x8)

        x9 = self.uptrans4(x8)
        x9 = torch.cat([x1, x9], dim=1)
        x9 = self.conv_up4(x9)

        # خروجی نهایی
        out = self.output(x9)
        return out

In [5]:
def dice_coefficient(pred, target, epsilon=1e-6):
    """
    محاسبه ضریب Dice برای ارزیابی مدل در سگمنتیشن باینری.
    pred و target دارای ابعاد N,C,H,W هستند.
    ما معمولا C=1 داریم (سگمنت تک کاناله).
    """
    pred = torch.sigmoid(pred)  # تبدیل خروجی شبکه به [0,1]
    pred = (pred > 0.5).float() # باینری کردن

    intersection = (pred * target).sum(dim=(1,2,3))
    union = pred.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) + epsilon
    dice = (2.0 * intersection + epsilon) / union
    return dice.mean()

In [7]:
import random
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

# خواندن لیست فایل‌ها
images_list = sorted(glob.glob(os.path.join(images_dir, "*.*")))
masks_list = sorted(glob.glob(os.path.join(labels_dir, "*.*")))
assert len(images_list) == len(masks_list), "تعداد تصاویر با ماسک‌ها همخوانی ندارد!"

# تقسیم داده‌ها به train، validation و test
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

total_size = len(images_list)
train_size = int(total_size * train_ratio)
val_size = int(total_size * val_ratio)
test_size = total_size - train_size - val_size

train_images, val_images, test_images = images_list[:train_size], images_list[train_size:train_size+val_size], images_list[train_size+val_size:]
train_masks, val_masks, test_masks = masks_list[:train_size], masks_list[train_size:train_size+val_size], masks_list[train_size+val_size:]

# تغییر اندازه‌ی تصاویر به 256×256
transform_resize = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# فرض می‌کنیم کلاس CorneaDataset و مدل UNet و تابع dice_coefficient قبلاً تعریف شده‌اند.
train_dataset = CorneaDataset(train_images, train_masks, transform=transform_resize)
val_dataset = CorneaDataset(val_images, val_masks, transform=transform_resize)
test_dataset = CorneaDataset(test_images, test_masks, transform=transform_resize)

batch_size = 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

# ساخت مدل
model = UNet(in_channels=3, out_channels=1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 15

torch.cuda.empty_cache()

# for epoch in range(num_epochs):
#     # آموزش مدل
#     model.train()
#     train_loss_sum = 0.0
#     train_dice_sum = 0.0
#     num_train_batches = 0
# 
#     for imgs, masks in train_loader:
#         imgs, masks = imgs.to(device), masks.to(device)
#         optimizer.zero_grad()
#         outputs = model(imgs)
#         loss = criterion(outputs, masks)
#         loss.backward()
#         optimizer.step()
#         dice_val = dice_coefficient(outputs, masks).item()
#         
#         train_loss_sum += loss.item()
#         train_dice_sum += dice_val
#         num_train_batches += 1
# 
#     avg_train_loss = train_loss_sum / num_train_batches
#     avg_train_dice = train_dice_sum / num_train_batches
# 
#     # ارزیابی روی داده‌های validation
#     model.eval()
#     val_loss_sum = 0.0
#     val_dice_sum = 0.0
#     num_val_batches = 0
#     with torch.no_grad():
#         for imgs, masks in val_loader:
#             imgs, masks = imgs.to(device), masks.to(device)
#             outputs = model(imgs)
#             loss = criterion(outputs, masks)
#             dice_val = dice_coefficient(outputs, masks).item()
#             
#             val_loss_sum += loss.item()
#             val_dice_sum += dice_val
#             num_val_batches += 1
#     avg_val_loss = val_loss_sum / num_val_batches
#     avg_val_dice = val_dice_sum / num_val_batches
# 
#     # ارزیابی روی داده‌های test
#     test_loss_sum = 0.0
#     test_dice_sum = 0.0
#     num_test_batches = 0
#     with torch.no_grad():
#         for imgs, masks in test_loader:
#             imgs, masks = imgs.to(device), masks.to(device)
#             outputs = model(imgs)
#             loss = criterion(outputs, masks)
#             dice_val = dice_coefficient(outputs, masks).item()
#             
#             test_loss_sum += loss.item()
#             test_dice_sum += dice_val
#             num_test_batches += 1
#     avg_test_loss = test_loss_sum / num_test_batches
#     avg_test_dice = test_dice_sum / num_test_batches
# 
#     # چاپ نتایج هر epoch
#     print(f"Epoch {epoch+1}/{num_epochs}")
#     print(f"Train   - Loss: {avg_train_loss:.4f}, Dice: {avg_train_dice:.4f}")
#     print(f"Validation - Loss: {avg_val_loss:.4f}, Dice: {avg_val_dice:.4f}")
#     print(f"Test    - Loss: {avg_test_loss:.4f}, Dice: {avg_test_dice:.4f}")
# 
# # ذخیره مدل
# model_path = "unet_cornea_segmentation.pth"
# torch.save(model.state_dict(), model_path)
# print(f"مدل ذخیره شد در: {model_path}")

In [16]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import os
import random
import matplotlib.pyplot as plt

# تنظیمات دستگاه (CPU یا GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# بارگذاری مدل ذخیره شده
model = UNet(in_channels=3, out_channels=1).to(device)
model.load_state_dict(torch.load("unet_cornea_segmentation.pth"))
model.eval()

# مسیر تصاویر بدون ماسک و مسیر ماسک‌های واقعی
images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

# دریافت لیست تصاویر موجود در مسیر
image_files = sorted(os.listdir(images_dir))
image_files = [os.path.join(images_dir, f) for f in image_files if f.endswith(('.jpg', '.png', '.jpeg'))]

# انتخاب 10 تصویر تصادفی از داده‌های ورودی
random_images = random.sample(image_files, 10)

# پیش‌پردازش تصویر (تغییر اندازه به 256x256 و تبدیل به Tensor)
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# نمایش خروجی مدل برای 10 تصویر تصادفی
fig, axes = plt.subplots(10, 3, figsize=(12, 30))  # اضافه کردن ستون سوم برای ماسک واقعی

for i, img_path in enumerate(random_images):
    img = Image.open(img_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)

    # پیش‌بینی با مدل
    with torch.no_grad():
        output = model(img_tensor)
        output = torch.sigmoid(output).squeeze().cpu().numpy()  # تبدیل خروجی به تصویر با مقادیر بین 0 و 1

    # پیدا کردن مسیر ماسک واقعی (با تغییر پسوند به .png)
    mask_name = os.path.basename(img_path)  # نام فایل تصویر
    mask_name = os.path.splitext(mask_name)[0] + ".png"  # تغییر پسوند به .png
    mask_path = os.path.join(labels_dir, mask_name)  # پیدا کردن مسیر ماسک واقعی

    # بارگذاری ماسک واقعی
    mask = Image.open(mask_path).convert('L')
    mask = transform(mask).unsqueeze(0).squeeze().cpu().numpy()

    # نمایش تصویر ورودی، ماسک واقعی و ماسک پیش‌بینی‌شده
    axes[i, 0].imshow(np.array(img), cmap='gray')
    axes[i, 0].set_title(f"Input Image {i+1}")
    axes[i, 0].axis('off')

    axes[i, 1].imshow(mask, cmap='gray')
    axes[i, 1].set_title(f"Real Mask {i+1}")
    axes[i, 1].axis('off')

    axes[i, 2].imshow(output, cmap='gray')
    axes[i, 2].set_title(f"Segmented {i+1}")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

In [21]:
import matplotlib.pyplot as plt
import random

# انتخاب ۱۰ نمونه از test_dataset (یا هر دیتاستی که مد نظر شماست)
num_samples = 10
# اگر می‌خواهید نمونه‌ها به صورت تصادفی انتخاب شوند:
indices = random.sample(range(len(test_dataset)), num_samples)
# یا به صورت ساده، ۱۰ نمونه اول:
# indices = list(range(num_samples))

fig, axes = plt.subplots(num_samples, 2, figsize=(8, 4*num_samples))
for i, idx in enumerate(indices):
    input_img, label_mask = test_dataset[idx]
    
    # تبدیل tensor ورودی از (3,256,256) به آرایه numpy با ابعاد (256,256,3)
    img_np = input_img.permute(1, 2, 0).numpy()
    # تبدیل ماسک از (1,256,256) به (256,256)
    mask_np = label_mask.squeeze(0).numpy()
    
    axes[i, 0].imshow(img_np)
    axes[i, 0].axis("off")
    axes[i, 0].set_title("input image")
    
    axes[i, 1].imshow(mask_np, cmap="gray")
    axes[i, 1].axis("off")
    axes[i, 1].set_title("mask")
    
plt.tight_layout()
plt.show()


In [22]:
import matplotlib.pyplot as plt
import random

# انتخاب ۱۰ نمونه از test_dataset (یا هر دیتاستی که مد نظر شماست)
num_samples = 10
indices = random.sample(range(len(test_dataset)), num_samples)

fig, axes = plt.subplots(num_samples, 2, figsize=(8, 4*num_samples))
for i, idx in enumerate(indices):
    input_img, label_mask = test_dataset[idx]
    
    # تبدیل tensor ورودی از (3,256,256) به آرایه numpy با ابعاد (256,256,3)
    img_np = input_img.permute(1, 2, 0).numpy()
    # تبدیل ماسک از (1,256,256) به (256,256)
    mask_np = label_mask.squeeze(0).numpy()
    
    # معکوس کردن ماسک: پس زمینه سفید به مشکی و ماسک مشکی به سفید
    mask_np = 1 - mask_np
    
    axes[i, 0].imshow(img_np)
    axes[i, 0].axis("off")
    axes[i, 0].set_title("input image")
    
    axes[i, 1].imshow(mask_np, cmap="gray")
    axes[i, 1].axis("off")
    axes[i, 1].set_title("mask")
    
plt.tight_layout()
plt.show()
